# Gosmart.Health GRAPHITE: Library Exploration Notebook

This interactive notebook demonstrates the core libraries and custom services used across the backend:
- **Configuration & Pydantic**: Pydantic Settings and data validation (`pydantic`, `pydantic-settings`)
- **PyDicom**: Reading, creating, and parsing DICOM datasets (`pydicom`)
- **Neo4j**: Graph database driver and Cypher query execution (`neo4j`)
- **Pykka**: Concurrent actor system for asynchronous processing (`pykka`)
- **PyNetDicom**: DICOM networking and C-ECHO verification (`pynetdicom`)
- **MCP Tools**: Model Context Protocol over SSE and direct tool calls (`mcp`)
- **CLI**: Command-line management via `gosmart-graphite`

## 1. Environment and Library Imports

In [ ]:
import sys
from pathlib import Path

# Ensure project root and src/ are in sys.path
PROJECT_ROOT = Path("..").resolve()
SRC_DIR = PROJECT_ROOT / "src"
for _p in (SRC_DIR, PROJECT_ROOT):
    if str(_p) not in sys.path:
        sys.path.insert(0, str(_p))

print(f"Python Version: {sys.version}")
print(f"Project Root: {PROJECT_ROOT}")

# Import core stack
import mcp

print("All core dependencies imported successfully!")

## 2. Configuration & Environment Variables (`GOSMART_GS_*`)

In [ ]:
from gosmart_graphite.core.config import get_settings

settings = get_settings()
print(f"App Name:          {settings.APP_NAME}")
print(f"App Version:       {settings.APP_VERSION}")
print(f"DB Connection URI: {settings.DB_CONNECTION_URI}")
print(f"Storage Root URI:  {settings.STORAGE_ROOT_URI}")
print(f"HTTP Server:       {settings.HOST}:{settings.PORT}")
print(f"DICOM AE Title:    {settings.DICOM_AE_TITLE} (Port {settings.DICOM_SCP_PORT})")

## 3. PyDicom: Dataset Creation & Parsing into Pydantic Hierarchy

In [ ]:
from gosmart_graphite.service.dicom_service import dicom_service

# Create an in-memory sample DICOM dataset
mock_ds = dicom_service.create_mock_dicom(
    patient_id="PT-10029",
    patient_name="ALICE^DOE",
    modality="MR",
    study_description="Brain MRI T1 Sagittal 3mm"
)

print("Generated DICOM Dataset:")
print(f"  Patient ID:         {mock_ds.PatientID}")
print(f"  Patient Name:       {mock_ds.PatientName}")
print(f"  Modality:           {mock_ds.Modality}")
print(f"  Study Instance UID: {mock_ds.StudyInstanceUID}")
print(f"  SOP Instance UID:   {mock_ds.SOPInstanceUID}")

# Parse using DicomService into structured Pydantic hierarchy
hierarchy = dicom_service.parse_dataset(mock_ds)
print("\nParsed Pydantic Hierarchy Model:")
print(hierarchy.model_dump_json(indent=2))

## 4. Pykka: Concurrency & Actor Messaging

In [ ]:
from gosmart_graphite.actor.dicom_actor import get_dicom_actor

# Retrieve or start the Pykka Actor
actor_ref = get_dicom_actor()
print(f"Actor URN: {actor_ref.actor_urn}")
print(f"Actor Alive: {actor_ref.is_alive()}")

# Ping the actor using ask() for synchronous request-response
ping_response = actor_ref.ask({"type": "PING"})
print(f"Actor Ping Response: {ping_response}")

# Process a mock DICOM generation via actor
ingest_result = actor_ref.ask({
    "type": "INGEST_MOCK",
    "patient_id": "PT-PYKKA-01",
    "modality": "CT"
})
print(f"Actor Ingestion Result: {ingest_result}")

## 5. Neo4j Graph Database Driver & Cypher

In [ ]:
from gosmart_graphite.db.neo4j import neo4j_client
from gosmart_graphite.service.graph_service import graph_service


async def test_neo4j():
    connected = await neo4j_client.verify_connectivity()
    print(f"Neo4j reachable at {settings.DB_CONNECTION_URI}: {connected}")
    if connected:
        records = await graph_service.execute_custom_query("""
            MATCH (n) 
            RETURN labels(n) AS labels, count(n) AS count
        """)
        print(f"Graph Node Counts: {records}")
    else:
        print("Note: Start Neo4j (e.g. docker run -p 7687:7687 -p 7474:7474 neo4j:latest) to run live Cypher queries.")

# In notebook, run async helper
await test_neo4j()

## 6. PyNetDicom: Application Entity (AE) & Network Connectivity

In [ ]:
from gosmart_graphite.service.pynetdicom_service import pynetdicom_service

ae = pynetdicom_service.get_ae()
print(f"Application Entity Title: {ae.ae_title}")
print(f"Requested Presentation Contexts: {len(ae.requested_contexts)}")
print(f"Supported Presentation Contexts: {len(ae.supported_contexts)}")

# Test C-ECHO (will return rejection if no SCP is listening on port 11112, but demonstrates full association handshake)
echo_res = pynetdicom_service.echo_scu(host="127.0.0.1", port=settings.DICOM_SCP_PORT)
print(f"C-ECHO Ping Test Result: {echo_res}")

## 7. Model Context Protocol (MCP) Tools

The server exposes MCP tools for agentic pair-programming and AI workflows.
Tools are served over SSE at `/mcp/sse` when the backend runs, and can also be queried programmatically:

In [ ]:
from gosmart_graphite.service.mcp_service import mcp

# List registered MCP tools
tools = await mcp.list_tools()
print(f"Registered MCP Tools ({len(tools)}):\n")
for t in tools:
    print(f"  • {t.name}: {t.description}")

# Call an MCP tool directly
health_result = await mcp.call_tool("health_check", {})
print(f"\nMCP Health Check Result:\n{health_result}")

## 8. CLI Management (`gosmart-graphite`)

The server is managed via the terminal CLI without requiring a web UI:
```bash
# Start server with HTTP API and MCP SSE endpoints
uv run gosmart-graphite start --port 8000

# Check health
uv run gosmart-graphite health

# Generate mock DICOM study into storage & Neo4j
uv run gosmart-graphite mock --patient-id "PT-100" --modality "CT"

# Ingest local DICOM file
uv run gosmart-graphite ingest /path/to/scan.dcm

# Query patients
uv run gosmart-graphite patients --limit 10

# Execute arbitrary Cypher
uv run gosmart-graphite query "MATCH (p:Patient) RETURN p.patient_id"

# Run standalone MCP server over SSE or stdio
uv run gosmart-graphite mcp --transport sse --port 8001
```